# 07 - Modular Pipeline Design and Execution

## Objective

This notebook demonstrates the reusable Python pipeline developed for processing Airbnb city datasets.

The pipeline separates ingestion, validation, cleaning, enrichment, and master-dataset construction into modular components.

The objective is to show that the same processing logic can be applied to multiple cities with minimal code changes.

## 1. Project Module Configuration

The reusable pipeline modules are stored in the project's `src` directory.

This notebook adds the `src` directory to the Python path so that ingestion, validation, cleaning, transformation, and orchestration modules can be imported and tested directly.

This structure separates reusable business logic from notebook-specific exploratory code.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

print("Project root:", PROJECT_ROOT)
print("SRC path:", SRC_PATH)
print("SRC exists:", SRC_PATH.exists())

Project root: c:\Users\Sagar\Documents\expernetic-airbnb-data-engineering\expernetic-airbnb-data-engineering
SRC path: c:\Users\Sagar\Documents\expernetic-airbnb-data-engineering\expernetic-airbnb-data-engineering\src
SRC exists: True


## 2. Modular Data Ingestion

The ingestion module loads the raw listings, reviews, and calendar files for a selected city.

The module accepts a city folder name as input and returns the corresponding source datasets.

This allows the same ingestion logic to process London, New York, or another city that follows the expected raw-data folder structure.

In [2]:
from ingestion.load_data import load_city_data

In [3]:
listings, reviews, calendar = load_city_data("london")

print("Listings:", listings.shape)
print("Reviews:", reviews.shape)
print("Calendar:", calendar.shape)

Listings: (96871, 79)
Reviews: (2097996, 6)
Calendar: (35357974, 7)


### Ingestion Validation Result

The London source datasets were loaded successfully.

The pipeline correctly retrieves:

- One listing-level dataset
- One review-level dataset
- One calendar-level dataset

The different dataset sizes reflect the one-to-many relationships between listings, reviews, and daily calendar records.

## 3. Data Validation

The validation module performs basic quality checks before transformation begins.

The current validation checks confirm that:

- Each source dataset is not empty
- The listings dataset contains the required listing identifier
- The reviews dataset contains `listing_id`
- The calendar dataset contains `listing_id`

These checks prevent the pipeline from continuing when essential source fields are missing.

In [4]:
from profiling.validate_data import validate_data

validate_data(
    listings,
    reviews,
    calendar
)

Running validation...
Validation passed.


## 4. Reusable Cleaning Module

The cleaning module applies standardized transformations to a city's listings dataset.

The module performs the following actions:

- Converts source price values into `price_clean`
- Calculates a dynamic 99th-percentile price-outlier threshold
- Creates the `price_outlier` flag
- Converts relevant date fields into datetime format
- Derives `host_tenure_years` where valid source data is available

The dynamic outlier threshold allows the same logic to adapt to different city price distributions without hardcoded values.

In [5]:
from cleaning.clean_listings import clean_listings

cleaned_listings = clean_listings(listings)

print(cleaned_listings[[
    "price",
    "price_clean",
    "price_outlier",
    "price_outlier_threshold",
    "host_tenure_years"
]].head())

     price  price_clean  price_outlier  price_outlier_threshold  \
0   $70.00         70.0          False                   1100.0   
1  $149.00        149.0          False                   1100.0   
2  $411.00        411.0          False                   1100.0   
3      NaN          NaN          False                   1100.0   
4  $210.00        210.0          False                   1100.0   

   host_tenure_years  
0          15.843836  
1          15.791781  
2          15.709589  
3          15.983562  
4          15.315068  


## 5. Review Aggregation Module

The review aggregation module transforms guest-review records into listing-level metrics.

For each listing, the module calculates:

- Total review count
- First recorded review date
- Last recorded review date

This aggregation prevents row multiplication when review data is later joined to the listing-level dataset.

In [6]:
from transformation.review_aggregation import aggregate_reviews

review_summary = aggregate_reviews(reviews)

print(review_summary.shape)
print(review_summary.head())

(72749, 4)
   listing_id  total_reviews first_review_date last_review_date
0       13913             55        2010-08-18       2025-08-21
1       15400             97        2009-12-21       2025-04-05
2       17402             56        2011-03-21       2024-02-19
3       24328             95        2010-11-15       2025-07-05
4       36274             15        2011-03-20       2025-09-06


## 6. Calendar Aggregation Module

The calendar aggregation module transforms daily calendar records into listing-level availability metrics.

For each listing, the module calculates:

- Total observed calendar days
- Available days
- Unavailable days
- Availability rate
- Occupancy/unavailability proxy

The occupancy proxy is based on the proportion of calendar dates marked unavailable. It should not be interpreted as confirmed booking occupancy because unavailable dates may also include host-blocked dates or other listing restrictions.

In [7]:
from transformation.calendar_aggregation import aggregate_calendar

calendar_summary = aggregate_calendar(calendar)

print(calendar_summary.shape)
print(calendar_summary.head())

(96871, 6)
   listing_id  total_days  available_days  unavailable_days  \
0       13913         365             331                34   
1       15400         365             199               166   
2       17402         365              80               285   
3       24328         365             294                71   
4       36274         365             323                42   

   availability_rate  occupancy_rate  
0           0.906849        0.093151  
1           0.545205        0.454795  
2           0.219178        0.780822  
3           0.805479        0.194521  
4           0.884932        0.115068  


## 7. Master Dataset Construction Module

The master dataset construction module combines cleaned listings data with the aggregated review and calendar summaries.

The module uses left joins to preserve every listing from the cleaned listings dataset, including listings with no review history.

The result is one enriched row per listing containing price, review, availability, and city-level context.

In [8]:
from transformation.build_master import build_master_dataset

city_master = build_master_dataset(
    cleaned_listings,
    review_summary,
    calendar_summary,
    "London"
)

print(city_master.shape)
print(city_master[[
    "city",
    "id",
    "price_clean",
    "total_reviews",
    "availability_rate",
    "occupancy_rate"
]].head())

(96871, 92)
     city     id  price_clean  total_reviews  availability_rate  \
0  London  13913         70.0           55.0           0.906849   
1  London  15400        149.0           97.0           0.545205   
2  London  17402        411.0           56.0           0.219178   
3  London  24328          NaN           95.0           0.805479   
4  London  36274        210.0           15.0           0.884932   

   occupancy_rate  
0        0.093151  
1        0.454795  
2        0.780822  
3        0.194521  
4        0.115068  


### Master Dataset Validation Result

The enriched city-level master dataset preserves one row per listing.

The output includes cleaned listing attributes, review-summary metrics, calendar-summary metrics, and the city identifier.

This confirms that the review and calendar datasets were correctly aggregated before joining and that the listing-level grain was maintained.

## 8. End-to-End Pipeline Orchestration

The `run_city_pipeline` function orchestrates the complete city-processing workflow.

The sequence is:

1. Load raw city files
2. Validate required source structures
3. Clean the listings dataset
4. Aggregate reviews to listing level
5. Aggregate calendar records to listing level
6. Build the enriched city master dataset
7. Save the processed output

This orchestration function allows the same pipeline to run for different city folders with minimal code changes.

In [9]:
from pipeline import run_city_pipeline

london_pipeline_master = run_city_pipeline(
    city_folder="london",
    city_name="London"
)

print(london_pipeline_master.shape)


Running validation...
Validation passed.
Saved: data\processed\london_master.csv
(96871, 92)


In [10]:
print(london_pipeline_master.head())

      id                         listing_url       scrape_id last_scraped  \
0  13913  https://www.airbnb.com/rooms/13913  20250914034649   2025-09-16   
1  15400  https://www.airbnb.com/rooms/15400  20250914034649   2025-09-16   
2  17402  https://www.airbnb.com/rooms/17402  20250914034649   2025-09-16   
3  24328  https://www.airbnb.com/rooms/24328  20250914034649   2025-09-18   
4  36274  https://www.airbnb.com/rooms/36274  20250914034649   2025-09-15   

            source                                               name  \
0      city scrape                Holiday London DB Room Let-on going   
1      city scrape                Bright Chelsea  Apartment. Chelsea!   
2      city scrape   Very Central Modern 3-Bed/2 Bath By Oxford St W1   
3  previous scrape                   Battersea live/work artist house   
4      city scrape  Bright 1 bedroom apt off brick lane in Shoreditch   

                                         description  \
0  My bright double bedroom with a large w

## 9. Multi-City Pipeline Execution

The same reusable pipeline is executed for New York using a different city folder and city label.

This demonstrates that the pipeline is configurable across multiple cities without rewriting ingestion, cleaning, enrichment, or validation logic.

In [11]:
from pipeline import run_city_pipeline

ny_pipeline_master = run_city_pipeline(
    city_folder="new_york",
    city_name="New York"
)

print(ny_pipeline_master.shape)


Running validation...
Validation passed.
Saved: data\processed\new_york_master.csv
(35036, 103)


In [12]:
from pipeline import run_city_pipeline

london_pipeline_master = run_city_pipeline("london", "London")
print(london_pipeline_master.shape)



Running validation...
Validation passed.
Saved: data\processed\london_master.csv
(96871, 92)


## 10. Cross-City Master Dataset Creation

The city-level pipeline outputs are combined into a single cross-city master dataset.

Each record retains a city label so that later SQL analytics, dimensional modeling, machine learning segmentation, and dashboard visualizations can distinguish between London and New York.

The final combined dataset remains at one row per listing.

In [13]:
from pipeline import run_city_pipeline
import pandas as pd

london_pipeline_master = run_city_pipeline("london", "London")
ny_pipeline_master = run_city_pipeline("new_york", "New York")

combined_pipeline_master = pd.concat(
    [london_pipeline_master, ny_pipeline_master],
    ignore_index=True
)

combined_pipeline_master.to_csv(
    "../data/processed/pipeline_master_dataset.csv",
    index=False
)

print(combined_pipeline_master.shape)
print(combined_pipeline_master["city"].value_counts())

Running validation...
Validation passed.
Saved: data\processed\london_master.csv
Running validation...
Validation passed.
Saved: data\processed\new_york_master.csv
(131907, 103)
city
London      96871
New York    35036
Name: count, dtype: int64


### Pipeline Execution Result

The reusable pipeline successfully processed both London and New York.

The combined output contains 131,907 listings:

- London: 96,871 listings
- New York: 35,036 listings

This demonstrates that the modular workflow can process multiple city datasets and create a unified analytical output using consistent transformation logic.

## 11. Current Pipeline Scope and Production Considerations

The implemented pipeline provides a modular and reusable local processing foundation.

The current implementation includes:

- Configurable city-based ingestion
- Basic validation checks
- Standardized cleaning logic
- Review and calendar aggregation
- Reusable master-dataset construction
- Multi-city processing

The following production capabilities are not yet implemented and are identified as future improvements:

- Structured logging
- Retry and exception-handling framework
- Incremental processing
- Metadata tracking
- Automated data lineage catalog
- Automated test suite
- Cloud deployment and orchestration

## Pipeline Design Summary and Next Steps

The project implements a modular local data pipeline that processes Airbnb source files into enriched listing-level outputs.

Key outcomes:

1. Reusable ingestion, validation, cleaning, and transformation modules were created.
2. The pipeline can process multiple cities with minimal code changes.
3. Review and calendar records are aggregated before joining to preserve one row per listing.
4. Processed city-level outputs are combined into a unified cross-city dataset.
5. The pipeline provides the foundation for dimensional modeling, SQL analytics, machine learning segmentation, and dashboard reporting.
6. Production-readiness gaps have been documented transparently for future implementation.

The next notebook applies unsupervised machine learning and guided analytics to identify Airbnb listing segments and provide market-intelligence insights.